In [2]:
"""
This notebook configures and runs a fixed-aggregation individual identification experiment
using precomputed audio embeddings. It supports multiple datasets, two embedding backbones,
and two fixed aggregation strategies through a modular configuration block.

Main functionalities
- Sets a global random seed for reproducibility using Lightning.
- Defines dataset, embedding, and output paths relative to the project root.
- Supports two embedding backbones:
    1) BirdNET embeddings extracted from 3 s windows.
    2) Perch embeddings extracted from 5 s windows.
- Supports two fixed aggregation strategies at the vocalization level:
    1) onset: uses the first embedding of each vocalization.
    2) gap: applies global average pooling over all embeddings in the vocalization.
- Loads metadata and embedding parts required for training, validation, and testing.
- Configures logging, metrics, classification reports, and figure output directories.
- Specifies the core model hyperparameters for the downstream classifier.

Supported datasets
- chiffchaff: supports within-year and across-year splits.
- pipit: supports within-year and across-year splits.
- littleowl: supports across-year only.
- littlepenguin: supports within-year only and uses a single metadata CSV.
- kiwi: supports across-year only and uses a single metadata CSV.
- rtbc: supports across-year only and uses a single metadata CSV.

Key configuration variables
- SEED: Random seed used for reproducibility in splitting and training.
- DATASET_NAME: Dataset to use.
- SPLIT_TYPE: Evaluation subset to use. Options depend on the dataset.
- MODEL: Embedding backbone, either "birdnet" or "perch".
- AGGREGATION: Fixed aggregation strategy, either "onset" or "gap".
- EXPERIMENT_TAG: Descriptive tag used to name the current experiment.
- PARENT_DIR: Project root directory used to build all relative paths.

Notes
- In onset mode, only the first embedding of each vocalization is used.
- In gap mode, the embeddings of each vocalization are averaged before classification.
- Ensure that all input directories and metadata files exist before running the notebook.
"""

from lightning import seed_everything
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset
import torch
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, recall_score, balanced_accuracy_score, accuracy_score, roc_auc_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from lightning import Trainer
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from torch import nn
import torch.nn.functional as F
from lightning import LightningModule
import torchmetrics
from torchmetrics.classification import F1Score
import logging
from lightning import LightningDataModule
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter



#SEED = 23



SEEDS = [18, 13, 46, 123, 321]
MODELS = ["birdnet", "perch"]
AGGREGATIONS = ["onset", "gap"]



# ---- Dataset / experiment config ----
# DATASET_NAME = "chiffchaff"        # options: "chiffchaff", "pipit", "littleowl", "littlepenguin", "kiwi", "rtbc"
# SPLIT_TYPE = "withinyear"    # options depend on dataset
# MODEL = "birdnet"              # options: "birdnet", "perch"
# AGGREGATION = "onset"          # options: "onset", "gap"

# MODEL = MODEL.lower()
# AGGREGATION = AGGREGATION.lower()

# if MODEL not in {"birdnet", "perch"}:
#     raise ValueError(f"Unknown MODEL: {MODEL}. Use 'birdnet' or 'perch'.")

# if AGGREGATION not in {"onset", "gap"}:
#     raise ValueError(f"Unknown AGGREGATION: {AGGREGATION}. Use 'onset' or 'gap'.")



# ---- Dataset-specific configuration ----
DATASET_CONFIG = {
    # "chiffchaff": {
    #     "metadata_dir": "chiffchaff-fg",
    #     "label_key_col": "wavfilename",
    #     "allowed_splits": {"withinyear", "acrossyear"},
    #     "train_files": {
    #         "withinyear": "chiffchaff-withinyear-fg-trn.csv",
    #         "acrossyear": "chiffchaff-acrossyear-fg-trn.csv",
    #     },
    #     "test_files": {
    #         "withinyear": "chiffchaff-withinyear-fg-tst.csv",
    #         "acrossyear": "chiffchaff-acrossyear-fg-tst.csv",
    #     },
    # },
    # "pipit": {
    #     "metadata_dir": "pipit-fg",
    #     "label_key_col": "wavfilename",
    #     "allowed_splits": {"withinyear", "acrossyear"},
    #     "train_files": {
    #         "withinyear": "pipit-withinyear-fg-trn.csv",
    #         "acrossyear": "pipit-acrossyear-fg-trn.csv",
    #     },
    #     "test_files": {
    #         "withinyear": "pipit-withinyear-fg-tst.csv",
    #         "acrossyear": "pipit-acrossyear-fg-tst.csv",
    #     },
    # },
    # "littleowl": {
    #     "metadata_dir": "littleowl-fg",
    #     "label_key_col": "wavfilename",
    #     "allowed_splits": {"acrossyear"},
    #     "train_files": {
    #         "acrossyear": "littleowl-acrossyear-fg-trn.csv",
    #     },
    #     "test_files": {
    #         "acrossyear": "littleowl-acrossyear-fg-tst.csv",
    #     },
    # },
    # "littlepenguin": {
    #     "metadata_dir": "littlepenguin_metadata",
    #     "label_key_col": "file_name",
    #     "allowed_splits": {"withinyear"},
    #     "train_files": {
    #         "withinyear": "littlepenguin_metadata_corrected.csv",
    #     },
    #     "test_files": {
    #         "withinyear": None,
    #     },
    # },
    # "kiwi": {
    #     "metadata_dir": "KiwiTrimmed",
    #     "label_key_col": "file_name",
    #     "allowed_splits": {"acrossyear"},
    #     "train_files": {
    #         "acrossyear": "kiwi_metadata.csv",
    #     },
    #     "test_files": {
    #         "acrossyear": None,
    #     },
    # },
    "rtbc": {
        "metadata_dir": "rtbc_metadata",
        "label_key_col": "file_name",
        "allowed_splits": {"acrossyear"},
        "train_files": {
            "acrossyear": "rtbc_metadata.csv",
        },
        "test_files": {
            "acrossyear": None,
        },
    },
}




# if DATASET_NAME not in DATASET_CONFIG:
#     raise ValueError(
#         f"Unknown DATASET_NAME: {DATASET_NAME}. "
#         f"Choose from {list(DATASET_CONFIG.keys())}."
#     )

# cfg = DATASET_CONFIG[DATASET_NAME]

# if SPLIT_TYPE not in cfg["allowed_splits"]:
#     raise ValueError(
#         f"Dataset '{DATASET_NAME}' does not support split '{SPLIT_TYPE}'. "
#         f"Allowed splits: {sorted(cfg['allowed_splits'])}"
#     )

# ---- Model-specific configuration ----

def run_experiment(seed, dataset_name, split_type, model, aggregation):
    SEED = seed
    DATASET_NAME = dataset_name
    SPLIT_TYPE = split_type
    MODEL = model
    AGGREGATION = aggregation

    seed_everything(SEED, workers=True)
    EXPERIMENT_TAG = f"{MODEL}_{AGGREGATION}"
    PARENT_DIR = Path.cwd().parent

    MODEL_CONFIG = {
        "birdnet": {
            "embeddings_dir": "Embeddings_from_3sPadding",
            "embedding_dim": 1024,
        },
        "perch": {
            "embeddings_dir": "Embeddings_from_5sPadding",
            "embedding_dim": 1280,
        },
    }

    model_cfg = MODEL_CONFIG[MODEL]

    # ---- Input paths ----
    PARTS_DIR = (
        PARENT_DIR
        / "Output_files"
        / model_cfg["embeddings_dir"]
        / f"{DATASET_NAME}_parquet_parts"
    )

    METADATA_DIR = PARENT_DIR / "Output_metadata" / cfg["metadata_dir"]
    TRN_CSV = METADATA_DIR / cfg["train_files"][SPLIT_TYPE]

    test_file = cfg["test_files"][SPLIT_TYPE]
    TST_CSV = None if test_file is None else METADATA_DIR / test_file

    # ---- Data columns ----
    LABEL_KEY_COL = cfg["label_key_col"]
    EMB_KEY_COL = "file_name"

    # Optional quick subset of parts (0 = use all)
    LIMIT_N_PARTS = 0

    # ---- Output paths ----
    LOG_ROOT = PARENT_DIR / "logs"
    RUN_NAME = f"{DATASET_NAME}_{SPLIT_TYPE}_{EXPERIMENT_TAG}"
    LOG_DIR = LOG_ROOT / DATASET_NAME / SPLIT_TYPE
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    RESULTS_DIR = PARENT_DIR / "Results" / DATASET_NAME / SPLIT_TYPE
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    METRICS_DIR = RESULTS_DIR / "Metrics"
    METRICS_DIR.mkdir(parents=True, exist_ok=True)

    METRICS_FILE = METRICS_DIR / f"{RUN_NAME}_final_metrics.csv"
    CLASS_REPORT_TXT_FILE = METRICS_DIR / f"{RUN_NAME}_test_classification_report.txt"
    CLASS_REPORT_CSV_FILE = METRICS_DIR / f"{RUN_NAME}_test_classification_report.csv"

    FIGURES_DIR = RESULTS_DIR / f"{RUN_NAME}_figures"
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    # ---- Model hyperparameters ----
    EMBEDDING_DIM = model_cfg["embedding_dim"]
    HIDDEN_DIM = 256
    LEARNING_RATE = 1e-3

    print("DATASET_NAME:", DATASET_NAME)
    print("SPLIT_TYPE:", SPLIT_TYPE)
    print("MODEL:", MODEL)
    print("AGGREGATION:", AGGREGATION)
    print("PARTS_DIR:", PARTS_DIR)
    print("TRN_CSV:", TRN_CSV)
    print("TST_CSV:", TST_CSV)
    print("LOG_DIR:", LOG_DIR / RUN_NAME)
    print("METRICS_FILE:", METRICS_FILE)
    print("FIGURES_DIR:", FIGURES_DIR)
    print("EMBEDDING_DIM:", EMBEDDING_DIM)


    # ---- Load available CSVs (train required, test optional) ----
    csv_paths = []
    if 'TRN_CSV' in locals() and TRN_CSV is not None and Path(TRN_CSV).exists():
        csv_paths.append(TRN_CSV)
    if 'TST_CSV' in locals() and TST_CSV is not None and Path(TST_CSV).exists():
        csv_paths.append(TST_CSV)

    if not csv_paths:
        raise ValueError("No label CSVs found. Provide at least TRN_CSV.")

    dfs = [pd.read_csv(p) for p in csv_paths]
    combined_df = pd.concat(dfs, ignore_index=True)

    # ---- Choose schema: single-label vs one-hot ----

    # Case A: single label column present (e.g., 'class')
    SINGLE_LABEL_COLS = [c for c in ["class", "label", "individual_id"] if c in combined_df.columns]

    if SINGLE_LABEL_COLS:
        LABEL_VALUE_COL = SINGLE_LABEL_COLS[0]  # pick the first available
        # Build label column directly
        combined_df["label"] = combined_df[LABEL_VALUE_COL].astype(str)

        # Normalize filenames for robust matching with embeddings
        if LABEL_KEY_COL not in combined_df.columns:
            raise ValueError(f"Missing filename column '{LABEL_KEY_COL}' in label CSV(s).")
        combined_df[LABEL_KEY_COL] = combined_df[LABEL_KEY_COL].astype(str).str.lower()

        combined_labels_dict = dict(zip(combined_df[LABEL_KEY_COL], combined_df["label"]))
        label_keys = set(combined_labels_dict.keys())

        print(f"[Single-label schema] Using '{LABEL_VALUE_COL}' as labels.")
        print(f"Loaded {len(combined_df):,} label rows; unique labeled audio files: {len(label_keys):,}")
        print("Example mapping:", list(combined_labels_dict.items())[:3])

    else:
        # Case B: one-hot schema (your original logic)
        if LABEL_KEY_COL not in combined_df.columns:
            raise ValueError(f"Missing filename column '{LABEL_KEY_COL}' in label CSV(s).")

        indiv_cols = [c for c in combined_df.columns if c != LABEL_KEY_COL]
        if not indiv_cols:
            raise ValueError("No one-hot individual columns found in the labels CSVs.")

        row_sums = combined_df[indiv_cols].sum(axis=1)
        if (row_sums == 0).all():
            raise ValueError("All rows in the one-hot block are zeros; no labeled files detected.")

        # One-hot → single string label: the column name containing the 1
        combined_df["label"] = combined_df[indiv_cols].idxmax(axis=1)

        # Normalize filenames
        combined_df[LABEL_KEY_COL] = combined_df[LABEL_KEY_COL].astype(str).str.lower()

        combined_labels_dict = dict(zip(combined_df[LABEL_KEY_COL], combined_df["label"]))
        label_keys = set(combined_labels_dict.keys())

        print(f"[One-hot schema] Derived labels from one-hot columns (n={len(indiv_cols)}).")
        print(f"Loaded {len(combined_df):,} label rows; unique labeled audio files: {len(label_keys):,}")
        print("Example mapping:", list(combined_labels_dict.items())[:3])

    # Find Parquet parts
    parts = sorted(PARTS_DIR.glob("part_*.parquet"))
    if LIMIT_N_PARTS > 0:
        parts = parts[:LIMIT_N_PARTS]

    if not parts:
        raise FileNotFoundError(f"No parquet parts found in: {PARTS_DIR}")

    print(f"Found {len(parts)} parquet parts; first 3: {[p.name for p in parts[:3]]}")

    # ------------------------------------------------------------------
    # Load labeled embedding rows
    # ------------------------------------------------------------------
    kept_chunks = []

    for part in parts:
        df_part = pd.read_parquet(part)

        if EMB_KEY_COL not in df_part.columns:
            raise ValueError(f"Parquet part {part.name} missing column '{EMB_KEY_COL}'")

        # Normalize filename column for matching
        df_part[EMB_KEY_COL] = (
            df_part[EMB_KEY_COL]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        # Keep only rows whose file appears in the metadata
        df_part = df_part[df_part[EMB_KEY_COL].isin(label_keys)].copy()
        if df_part.empty:
            continue

        # Drop path if present
        if "path" in df_part.columns:
            df_part = df_part.drop(columns=["path"])

        kept_chunks.append(df_part)

    if not kept_chunks:
        raise ValueError("No embeddings matched the labeled filenames.")

    trainvaltest_df = pd.concat(kept_chunks, ignore_index=True)

    # Map labels
    trainvaltest_df["label"] = trainvaltest_df[EMB_KEY_COL].map(combined_labels_dict)

    missing_labels = trainvaltest_df["label"].isna().sum()
    if missing_labels > 0:
        raise ValueError(f"{missing_labels} rows have missing labels after mapping.")

    print(f"Rows before aggregation: {len(trainvaltest_df):,}")
    print(f"Unique files before aggregation: {trainvaltest_df[EMB_KEY_COL].nunique():,}")

    # ------------------------------------------------------------------
    # Apply file-level aggregation
    # ------------------------------------------------------------------
    if AGGREGATION == "onset":
        if "start_time" not in trainvaltest_df.columns:
            raise ValueError("The 'start_time' column is required for onset aggregation.")

        # Sort within each file and keep the first embedding
        trainvaltest_df = (
            trainvaltest_df
            .sort_values(by=[EMB_KEY_COL, "start_time"])
            .groupby([EMB_KEY_COL, "label"], as_index=False)
            .first()
        )

        print("Applied onset aggregation: kept the first embedding per file.")

    elif AGGREGATION == "gap":
        cols_to_group = [EMB_KEY_COL, "label"]

        # Average numeric columns per file
        trainvaltest_df = (
            trainvaltest_df
            .groupby(cols_to_group, as_index=False)
            .mean(numeric_only=True)
        )

        # Drop time columns after pooling if they remain
        cols_to_drop = [c for c in ["start_time", "end_time"] if c in trainvaltest_df.columns]
        if cols_to_drop:
            trainvaltest_df = trainvaltest_df.drop(columns=cols_to_drop)

        print("Applied GAP aggregation: averaged embeddings per file.")

    else:
        raise ValueError(
            f"Unknown AGGREGATION: {AGGREGATION}. Use 'onset' or 'gap'."
        )

    print(f"Rows after aggregation: {len(trainvaltest_df):,}")
    print(f"Unique files after aggregation: {trainvaltest_df[EMB_KEY_COL].nunique():,}")
    print("Final columns:", trainvaltest_df.columns.tolist())
    assert trainvaltest_df[EMB_KEY_COL].nunique() == len(trainvaltest_df), \
        "Aggregation failed: more than one row remains for at least one file."
    # Map file_name → label
    trainvaltest_df["label"] = trainvaltest_df[EMB_KEY_COL].map(combined_labels_dict)

    # (Optional) sanity: any missing labels after mapping?
    missing = trainvaltest_df["label"].isna().sum()
    if missing:
        print(f"Warning: {missing} rows had missing labels after mapping.")

    # Unique file-level table (one row per file with its label)
    file_labels = trainvaltest_df[[EMB_KEY_COL, "label"]].drop_duplicates()

    # 1) 80/20 split (train+val / test)
    train_val_files_only, test_files_only = train_test_split(
        file_labels,
        test_size=0.20,
        stratify=file_labels["label"],
        random_state=SEED+19,
    )

    # 2) Split the 80 into 60/20 (train / val)
    train_files_only, val_files_only = train_test_split(
        train_val_files_only,
        test_size=0.25,  # 25% of 80% → 20% total
        stratify=train_val_files_only["label"],
        random_state=SEED+19,
    )

    # 3) Build final row-level DataFrames by selecting rows for those file sets
    train_df = trainvaltest_df[trainvaltest_df[EMB_KEY_COL].isin(train_files_only[EMB_KEY_COL])].copy()
    val_df   = trainvaltest_df[trainvaltest_df[EMB_KEY_COL].isin(val_files_only[EMB_KEY_COL])].copy()
    test_df  = trainvaltest_df[trainvaltest_df[EMB_KEY_COL].isin(test_files_only[EMB_KEY_COL])].copy()

    train_df = train_df.sort_index()
    val_df   = val_df.sort_index()
    test_df  = test_df.sort_index()

    print(f"Train: {len(train_df):,} rows")
    print(f"Validation: {len(val_df):,} rows")
    print(f"Test: {len(test_df):,} rows")

    print("Labels in TRN:", sorted(train_df["label"].unique()))
    print("Labels in VAL:", sorted(val_df["label"].unique()))
    print("Labels in TST:", sorted(test_df["label"].unique()))

    le = LabelEncoder()

    # Fit on training labels only
    train_df["label"] = le.fit_transform(train_df["label"])

    # Transform val/test consistently
    val_df["label"] = le.transform(val_df["label"])
    test_df["label"] = le.transform(test_df["label"])

    print("Encoded classes:", list(le.classes_))



    def build_pooled_per_file_dicts(df, key_col, dims_cols, label_col):
        """
        Return (embeddings_dict, id_dict, file_list) from a pooled DataFrame.

        Assumes the input DataFrame already contains exactly one aggregated embedding
        vector per file, as produced by onset or GAP aggregation.
        """
        embeddings_dict = {}
        id_dict = {}

        for _, row in df.iterrows():
            file_name = row[key_col]
            vector = row[dims_cols].to_numpy(dtype="float32")  # shape (D,)

            embeddings_dict[file_name] = vector
            id_dict[file_name] = row[label_col]

        files = list(id_dict.keys())
        return embeddings_dict, id_dict, files


    # Identify embedding dimension columns
    dims_cols = [c for c in train_df.columns if c.startswith("dim_")]

    # Train
    audio_train_embeddings, id_train_dict, train_files = build_pooled_per_file_dicts(
        train_df, EMB_KEY_COL, dims_cols, "label"
    )

    # Val
    audio_val_embeddings, id_val_dict, val_files = build_pooled_per_file_dicts(
        val_df, EMB_KEY_COL, dims_cols, "label"
    )

    # Test
    audio_test_embeddings, id_test_dict, test_files = build_pooled_per_file_dicts(
        test_df, EMB_KEY_COL, dims_cols, "label"
    )


    class SubsetAudioDataset(Dataset):
        """
        Return (embedding, label) for a subset of files.

        Parameters
        ----------
        file_list : list
            List of file names belonging to the split.
        audio_embeddings : dict
            Dictionary mapping file_name -> pooled embedding vector of shape (D,).
        file_to_label : dict
            Dictionary mapping file_name -> encoded class label.
        """
        def __init__(self, file_list, audio_embeddings, file_to_label):
            self.file_list = file_list
            self.audio_embeddings = audio_embeddings
            self.file_to_label = file_to_label

        def __len__(self):
            return len(self.file_list)

        def __getitem__(self, idx):
            file_name = self.file_list[idx]
            emb_np = self.audio_embeddings[file_name]  # shape (D,)

            if emb_np.ndim != 1:
                raise ValueError(
                    f"Expected a 1D pooled embedding for file '{file_name}', "
                    f"but got shape {emb_np.shape}"
                )

            embeddings = torch.tensor(emb_np, dtype=torch.float32)  # shape (D,)
            label = torch.tensor(self.file_to_label[file_name], dtype=torch.long)

            return embeddings, label


    # Create datasets
    train_dataset = SubsetAudioDataset(train_files, audio_train_embeddings, id_train_dict)
    val_dataset   = SubsetAudioDataset(val_files,   audio_val_embeddings,  id_val_dict)
    test_dataset  = SubsetAudioDataset(test_files,  audio_test_embeddings, id_test_dict)

    print(f"Train size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")


    # --- Count labels in each split ---
    def count_labels(dataset):
        """Count label frequencies in a PyTorch dataset."""
        return Counter(label.item() for _, label in dataset)

    datasets = {
        "train": train_dataset,
        "val": val_dataset,
        "test": test_dataset
    }

    # Create DataFrame with counts
    counts_df = pd.DataFrame({
        split: pd.Series(count_labels(ds))
        for split, ds in datasets.items()
    }).fillna(0).astype(int)

    # --- Sort by frequency in train ---
    counts_df = counts_df.sort_values(by="train", ascending=False)

    # --- Compute totals (not for plotting) ---
    total_per_individual = counts_df.sum(axis=1)

    # --- Metrics ---
    total_vocalizations = int(total_per_individual.sum())
    max_vocalizations   = int(total_per_individual.max())
    min_vocalizations   = int(total_per_individual.min())
    mean_vocalizations  = float(total_per_individual.mean())
    std_vocalizations   = float(total_per_individual.std(ddof=1))  # sample SD

    # --- Display counts DataFrame ---
    display(counts_df)

    # --- Print requested metrics ---
    print("=== Dataset metrics (all splits combined) ===")
    print(f"Total vocalizations: {total_vocalizations}")
    print(f"Maximum vocalizations per individual: {max_vocalizations}")
    print(f"Minimum vocalizations per individual: {min_vocalizations}")
    print(f"Vocalizations per individual (mean ± SD): {mean_vocalizations:.1f} ± {std_vocalizations:.1f}")
    print(f"Vocalizations per individual (mean ± SD, rounded): {round(mean_vocalizations):.0f} ± {round(std_vocalizations):.0f}")

    # --- Plot per-split distribution ---
    # counts_df.plot(kind='bar', figsize=(10, 6))
    # plt.xlabel('Individual (Label)')
    # plt.ylabel('Number of Samples')
    # plt.title('Distribution of Individuals in Train / Validation / Test')
    # plt.legend(title='Dataset')
    # plt.tight_layout()
    # plt.show()




    class PooledEmbeddingDataModule(LightningDataModule):
        def __init__(
            self,
            train_dataset,
            val_dataset,
            test_dataset,
            batch_size: int = 32,
            num_workers: int = 0,              # 0 is safest in notebooks
            pin_memory: bool = False,          # set True if training on GPU
            persistent_workers: bool = False,  # True only if num_workers > 0
            drop_last: bool = False,
        ):
            super().__init__()
            self.train_dataset = train_dataset
            self.val_dataset = val_dataset
            self.test_dataset = test_dataset

            self.batch_size = batch_size
            self.num_workers = num_workers
            self.pin_memory = pin_memory
            self.persistent_workers = persistent_workers if num_workers > 0 else False
            self.drop_last = drop_last

        def _make_loader(self, dataset, shuffle: bool):
            return DataLoader(
                dataset,
                batch_size=self.batch_size,
                shuffle=shuffle,
                num_workers=self.num_workers,
                pin_memory=self.pin_memory,
                persistent_workers=self.persistent_workers,
                drop_last=self.drop_last,
            )

        def train_dataloader(self):
            return self._make_loader(self.train_dataset, shuffle=True)

        def val_dataloader(self):
            return self._make_loader(self.val_dataset, shuffle=False)

        def test_dataloader(self):
            return self._make_loader(self.test_dataset, shuffle=False)


    use_cuda = torch.cuda.is_available()

    data_module = PooledEmbeddingDataModule(
        train_dataset,
        val_dataset,
        test_dataset,
        batch_size=32,
        num_workers=0,
        pin_memory=use_cuda,
        persistent_workers=False,
        drop_last=False,
    )


    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)


    class BirdsongClassifier(LightningModule):
        """
        Fully connected classifier for fixed-size embedding vectors of shape (B, D).
        """

        def __init__(
            self,
            embedding_dim: int,
            hidden_dim: int,
            num_classes: int,
            lr: float = 1e-3,
            dropout: float = 0.0,
            class_weights: torch.Tensor | None = None,
        ):
            super().__init__()
            self.save_hyperparameters(ignore=["class_weights"])
            self.class_weights = class_weights

            self.net = nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, num_classes),
            )

            self.train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
            self.val_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
            self.test_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

            self.val_f1 = torchmetrics.F1Score(
                task="multiclass",
                num_classes=num_classes,
                average="macro",
            )
            self.test_f1 = torchmetrics.F1Score(
                task="multiclass",
                num_classes=num_classes,
                average="macro",
            )

            self.criterion = nn.CrossEntropyLoss(weight=None)

        def setup(self, stage=None):
            if self.class_weights is not None:
                self.criterion = nn.CrossEntropyLoss(weight=self.class_weights.to(self.device))

        @staticmethod
        def _unpack_batch(batch):
            """
            Support both (x, y) and legacy (x, lengths, y) batches.
            Lengths are ignored for the FCN model.
            """
            if isinstance(batch, (list, tuple)):
                if len(batch) == 2:
                    x, y = batch
                    return x, y
                if len(batch) == 3:
                    x, _, y = batch
                    return x, y
            raise ValueError("Expected batch as (x, y) or (x, lengths, y)")

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            logits = self.net(x)  # (B, C)
            return logits

        def _shared_step(self, batch, stage: str):
            x, y = self._unpack_batch(batch)
            logits = self(x)
            loss = self.criterion(logits, y)

            if stage == "train":
                acc = self.train_accuracy(logits, y)
                self.log("train_loss", loss, prog_bar=True)
                self.log("train_acc", acc, prog_bar=True)
                return loss

            if stage == "val":
                acc = self.val_accuracy(logits, y)
                f1 = self.val_f1(logits, y)
                self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
                self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
                self.log("val_f1", f1, prog_bar=True, on_step=False, on_epoch=True)
                return loss

            if stage == "test":
                acc = self.test_accuracy(logits, y)
                f1 = self.test_f1(logits, y)
                self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
                self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
                self.log("test_f1", f1, prog_bar=True, on_step=False, on_epoch=True)
                return loss

            raise ValueError(f"Unknown stage: {stage}")

        def training_step(self, batch, batch_idx):
            loss = self._shared_step(batch, stage="train")
            logger.info(f"Batch {batch_idx} - Train Loss: {loss.item():.4f}")
            return loss

        def validation_step(self, batch, batch_idx):
            self._shared_step(batch, stage="val")

        def test_step(self, batch, batch_idx):
            loss = self._shared_step(batch, stage="test")
            logger.info(f"Batch {batch_idx} - Test Loss: {loss.item():.4f}")

        def configure_optimizers(self):
            return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
    all_labels = (
        list(id_train_dict.values()) +
        list(id_val_dict.values()) +
        list(id_test_dict.values())
    )

    model = BirdsongClassifier(
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=len(set(all_labels)),  # Adjusted based on label encoder
        lr=LEARNING_RATE
    )



    # Save the best model (by val_f1) and also keep the last checkpoint
    checkpoint_callback = ModelCheckpoint(
        monitor="val_f1",
        mode="max",
        save_top_k=1,
        save_last=True,
        filename=f"best-{DATASET_NAME}-{SPLIT_TYPE}-{EXPERIMENT_TAG}-{{epoch:02d}}-{{val_f1:.4f}}"
        # auto_insert_metric_name=False,  # uncomment if you don't want Lightning to append metric names
    )

    # Early stopping if validation doesn't improve
    early_stop_callback = EarlyStopping(
        monitor="val_f1",
        mode="max",
        patience=10,
        min_delta=1e-4,
    )



    # TensorBoard + CSV under: logs/<DATASET_NAME>/<SPLIT_TYPE>/<RUN_NAME>/version_x/
    tb_logger = TensorBoardLogger(
        save_dir=str(LOG_DIR),   # logs/<DATASET_NAME>/<SPLIT_TYPE>
        name=RUN_NAME,           # <DATASET_NAME>_<SPLIT_TYPE>_RNN
    )

    csv_logger = CSVLogger(
        save_dir=str(LOG_DIR),   # same root, same run name
        name=RUN_NAME,
    )

    trainer = Trainer(
        max_epochs=40,
        accelerator="auto",
        devices="auto",
        callbacks=[checkpoint_callback, early_stop_callback],
        logger=[tb_logger, csv_logger],
        log_every_n_steps=10,
        # Reproducibility and stability (safe defaults)
        deterministic=True,        # ensures repeatable results (may be slightly slower)
        # precision="16-mixed",    # optional: speed-up on GPU; leave commented to keep numerics identical
    )

    trainer.fit(model, datamodule=data_module)


    # Find the latest version directory created by the loggers
    run_dir = LOG_DIR / RUN_NAME
    version_dirs = sorted([p for p in run_dir.glob("version_*") if p.is_dir()],
                        key=lambda p: int(p.name.split("_")[-1]))
    if not version_dirs:
        raise FileNotFoundError(f"No version_* folders found under {run_dir}")
    latest = version_dirs[-1]
    metrics_csv = latest / "metrics.csv"
    print("Reading metrics from:", metrics_csv)

    df = pd.read_csv(metrics_csv)

    # Helper: pick the x-axis. Prefer 'step' if present, else 'epoch' (falls back to index).
    x = df["step"] if "step" in df.columns else (df["epoch"] if "epoch" in df.columns else df.index)

    # Filter rows that actually contain each metric
    df_train_loss = df[df["train_loss"].notna()] if "train_loss" in df.columns else pd.DataFrame()
    df_val_loss   = df[df["val_loss"].notna()]   if "val_loss"   in df.columns else pd.DataFrame()
    df_train_acc  = df[df["train_acc"].notna()]  if "train_acc"  in df.columns else pd.DataFrame()
    df_val_acc    = df[df["val_acc"].notna()]    if "val_acc"    in df.columns else pd.DataFrame()

    # ---- Plot 1: Train vs Validation Loss ----
    # plt.figure(figsize=(8, 5))
    # if not df_train_loss.empty:
    #     plt.plot(df_train_loss["step"] if "step" in df_train_loss else df_train_loss.index,
    #             df_train_loss["train_loss"], label="Train Loss")
    # if not df_val_loss.empty:
    #     plt.plot(df_val_loss["step"] if "step" in df_val_loss else df_val_loss.index,
    #             df_val_loss["val_loss"], label="Validation Loss")
    # plt.xlabel("Step")
    # plt.ylabel("Loss")
    # plt.title("Training & Validation Loss")
    # plt.legend()
    # plt.grid(True, linestyle="--", alpha=0.5)
    # plt.tight_layout()
    # plt.show()

    # ---- Plot 2: Train vs Validation Accuracy ----
    # plt.figure(figsize=(8, 5))
    # if not df_train_acc.empty:
    #     plt.plot(df_train_acc["step"] if "step" in df_train_acc else df_train_acc.index,
    #             df_train_acc["train_acc"], label="Train Accuracy")
    # if not df_val_acc.empty:
    #     plt.plot(df_val_acc["step"] if "step" in df_val_acc else df_val_acc.index,
    #             df_val_acc["val_acc"], label="Validation Accuracy")
    # plt.xlabel("Step")
    # plt.ylabel("Accuracy")
    # plt.title("Training & Validation Accuracy")
    # plt.legend()
    # plt.grid(True, linestyle="--", alpha=0.5)
    # plt.tight_layout()
    # plt.show()

    trainer.test(model, datamodule=data_module)
    # Load the best model (optional)
    best_model_path = checkpoint_callback.best_model_path
    print(f"Best model saved at: {best_model_path}")

    # Alternative: load a specific checkpoint manually
    # best_model_path = "KiWi_logs/individual_identification/version_3/checkpoints/best-individual-identification-KiWi.ckpt"

    best_model = BirdsongClassifier.load_from_checkpoint(best_model_path)
    best_model.eval()
    # Evaluate the best model on the test set
    results = trainer.test(best_model, datamodule=data_module)

    # Print all metrics in a nice format
    print("\nTest Metrics:")
    for metric, value in results[0].items():
        print(f"{metric}: {value:.4f} ({value*100:.1f}%)" if "acc" in metric or "f1" in metric else f"{metric}: {value:.4f}")

    test_acc = results[0]['test_acc']  



    # Load best checkpoint (if not already loaded)
    best_model = BirdsongClassifier.load_from_checkpoint(best_model_path)
    best_model.eval()

    # Put model on the right device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_model.to(device)

    all_preds, all_labels = [], []

    # Iterate over the test loader
    with torch.no_grad():
        for x, y in data_module.test_dataloader():  # <-- only (x, y) now
            x = x.to(device)
            y = y.to(device)

            logits = best_model(x)                  # <-- no lengths
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
            
    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)

    # Map class indices -> original string labels from your LabelEncoder
    labels = list(range(len(le.classes_)))
    target_names = [str(c) for c in le.classes_]

    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        digits=3,
        zero_division=0,   # avoids warnings if a class has no predicted samples
    )

    print(report_text)

    # save the report alongside TensorBoard logs
    with open(CLASS_REPORT_TXT_FILE, "w") as f:
        f.write(report_text)
    print(f"\nSaved classification report (txt) to: {CLASS_REPORT_TXT_FILE}")


    # Dictionary report for CSV export
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        digits=4,
        output_dict=True
    )
    report_df = pd.DataFrame(report_dict).transpose()

    # Optional: keep only per-individual rows
    per_class_report_df = report_df.loc[list(le.classes_)].reset_index()
    per_class_report_df = per_class_report_df.rename(columns={"index": "individual"})

    per_class_report_df.to_csv(CLASS_REPORT_CSV_FILE, index=False)

    print(f"Saved classification report (csv) to: {CLASS_REPORT_CSV_FILE}")
    display(per_class_report_df.head())



    def evaluate_split(model, loader, le, split_name: str):
        """Evaluate a model on a dataloader and compute multiple metrics."""
        model.eval()
        device = next(model.parameters()).device

        all_preds, all_labels, all_logits = [], [], []

        with torch.no_grad():
            for x, y in loader:                         # CHANGED: (x, y)
                x = x.to(device)
                y = y.to(device)

                logits = model(x)                       # CHANGED: model(x)
                preds = torch.argmax(logits, dim=1)

                all_logits.append(logits.detach().cpu().numpy())
                all_preds.append(preds.detach().cpu().numpy())
                all_labels.append(y.detach().cpu().numpy())

        # Arrays
        y_true = np.concatenate(all_labels)
        y_pred = np.concatenate(all_preds)
        logits_np = np.concatenate(all_logits, axis=0)

        # Confusion matrix (plot)
        labels = list(range(len(le.classes_)))
        # cm = confusion_matrix(y_true, y_pred, labels=labels)
        # fig, ax = plt.subplots(figsize=(8, 6))
        # ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_).plot(
        #     ax=ax, xticks_rotation=45, cmap="Blues", colorbar=True
        # )
        # ax.set_title(f"Confusion Matrix — {split_name}")
        # plt.tight_layout()
        # plt.show()

        # Metrics
        acc = accuracy_score(y_true, y_pred)
        acc_bal = balanced_accuracy_score(y_true, y_pred)
        recall_macro    = recall_score(y_true, y_pred, labels=labels, average="macro",    zero_division=0)
        recall_weighted = recall_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
        f1_macro        = f1_score(y_true, y_pred,     labels=labels, average="macro",    zero_division=0)
        f1_weighted     = f1_score(y_true, y_pred,     labels=labels, average="weighted", zero_division=0)

        # ROC AUC (macro)
        num_classes = len(np.unique(y_true))
        y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))
        probs = torch.softmax(torch.tensor(logits_np), dim=1).numpy()
        roc_auc_macro = roc_auc_score(y_true_bin, probs, average="macro", multi_class="ovr")

        # Print and return
        print(f"{split_name} — Accuracy:          {acc:.4f} ({acc*100:.1f}%)")
        print(f"{split_name} — Balanced accuracy: {acc_bal:.4f} ({acc_bal*100:.1f}%)")
        print(f"{split_name} — Recall (macro):    {recall_macro:.4f}")
        print(f"{split_name} — Recall (weighted): {recall_weighted:.4f}")
        print(f"{split_name} — F1 (macro):        {f1_macro:.4f} ({f1_macro*100:.1f}%)")
        print(f"{split_name} — F1 (weighted):     {f1_weighted:.4f}")
        print(f"{split_name} — ROC AUC (macro):   {roc_auc_macro:.4f}\n")

        return {
            "accuracy": acc,
            "balanced_accuracy": acc_bal,
            "recall_macro": recall_macro,
            "recall_weighted": recall_weighted,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
            "roc_auc_macro": roc_auc_macro,
        }


    # Ensure the model is on the right device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_model.to(device)

    # Evaluate and collect metrics
    val_metrics  = evaluate_split(best_model, data_module.val_dataloader(),  le, "Validation")
    test_metrics = evaluate_split(best_model, data_module.test_dataloader(), le, "Test")

    summary_row = {
        "run_name": RUN_NAME,
        "seed": SEED,
        "dataset_name": DATASET_NAME,
        "split_type": SPLIT_TYPE,
        "model": MODEL,
        "aggegration": AGGREGATION,
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "learning_rate": LEARNING_RATE,

        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_recall_macro": val_metrics["recall_macro"],
        "val_recall_weighted": val_metrics["recall_weighted"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_roc_auc_macro": val_metrics["roc_auc_macro"],

        "test_accuracy": test_metrics["accuracy"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_recall_macro": test_metrics["recall_macro"],
        "test_recall_weighted": test_metrics["recall_weighted"],
        "test_f1_macro": test_metrics["f1_macro"],
        "test_f1_weighted": test_metrics["f1_weighted"],
        "test_roc_auc_macro": test_metrics["roc_auc_macro"],
    }

    new_row_df = pd.DataFrame([summary_row])

    key_cols = ["run_name", "seed"]

    if METRICS_FILE.exists():
        metrics_df = pd.read_csv(METRICS_FILE)

        missing_cols = [c for c in key_cols if c not in metrics_df.columns]
        if missing_cols:
            raise ValueError(
                f"Missing key columns in existing metrics file: {missing_cols}"
            )

        mask = pd.Series(True, index=metrics_df.index)
        for col in key_cols:
            mask &= metrics_df[col] == summary_row[col]

        if mask.any():
            metrics_df = metrics_df.loc[~mask].copy()
            action = "updated"
        else:
            action = "appended"

        metrics_df = pd.concat([metrics_df, new_row_df], ignore_index=True)

    else:
        metrics_df = new_row_df.copy()
        action = "created"

    metrics_df = metrics_df.sort_values(by=key_cols).reset_index(drop=True)
    metrics_df.to_csv(METRICS_FILE, index=False)

    print(f"Metrics file {action}: {METRICS_FILE}")
    display(metrics_df)
    # ------------------------------------------------------------------
    # Aggregate summary over all seeds currently stored in METRICS_FILE
    # ------------------------------------------------------------------
    metrics_df = pd.read_csv(METRICS_FILE)

    metric_cols = ["test_accuracy", "test_f1_macro", "test_roc_auc_macro"]
    missing_metric_cols = [c for c in metric_cols if c not in metrics_df.columns]
    if missing_metric_cols:
        raise ValueError(f"Missing metric columns in metrics file: {missing_metric_cols}")

    aggregate_summary = {
        "dataset_name": metrics_df["dataset_name"].iloc[0],
        "split_type": metrics_df["split_type"].iloc[0],
        "model": metrics_df["model"].iloc[0],
        "aggegration_mode": metrics_df["aggegration"].iloc[0],
        "embedding_dim": metrics_df["embedding_dim"].iloc[0],
        "n_seeds": metrics_df["seed"].nunique(),

        "test_accuracy": f"{metrics_df['test_accuracy'].mean():.3f} ± {metrics_df['test_accuracy'].std(ddof=1):.3f}",
        "test_f1_macro": f"{metrics_df['test_f1_macro'].mean():.3f} ± {metrics_df['test_f1_macro'].std(ddof=1):.3f}",
        "test_roc_auc_macro": f"{metrics_df['test_roc_auc_macro'].mean():.4f} ± {metrics_df['test_roc_auc_macro'].std(ddof=1):.4f}",
    }

    aggregate_summary_df = pd.DataFrame([aggregate_summary])

    print("\nAggregate summary across seeds:")
    display(aggregate_summary_df)

for DATASET_NAME, cfg in DATASET_CONFIG.items():
    for SPLIT_TYPE in cfg["allowed_splits"]:
        for MODEL in MODELS:
            for AGGREGATION in AGGREGATIONS:
                for SEED in SEEDS:
                    run_experiment(
                        seed=SEED,
                        dataset_name=DATASET_NAME,
                        split_type=SPLIT_TYPE,
                        model=MODEL,
                        aggregation=AGGREGATION,
                    )


Seed set to 18


DATASET_NAME: rtbc
SPLIT_TYPE: acrossyear
MODEL: birdnet
AGGREGATION: onset
PARTS_DIR: /teamspace/studios/this_studio/Output_files/Embeddings_from_3sPadding/rtbc_parquet_parts
TRN_CSV: /teamspace/studios/this_studio/Output_metadata/rtbc_metadata/rtbc_metadata.csv
TST_CSV: None
LOG_DIR: /teamspace/studios/this_studio/logs/rtbc/acrossyear/rtbc_acrossyear_birdnet_onset
METRICS_FILE: /teamspace/studios/this_studio/Results/rtbc/acrossyear/Metrics/rtbc_acrossyear_birdnet_onset_final_metrics.csv
FIGURES_DIR: /teamspace/studios/this_studio/Results/rtbc/acrossyear/rtbc_acrossyear_birdnet_onset_figures
EMBEDDING_DIM: 1024
[Single-label schema] Using 'class' as labels.
Loaded 1,785 label rows; unique labeled audio files: 1,785
Example mapping: [('32pc1_2022-23_57.wav', '32PC1_2022-23'), ('32pc1_2022-23_0.wav', '32PC1_2022-23'), ('32pc1_2022-23_74.wav', '32PC1_2022-23')]
Found 2 parquet parts; first 3: ['part_0000.parquet', 'part_0001.parquet']
Rows before aggregation: 1,785
Unique files before ag

,train,val,test
8,233,78,77
9,187,62,62
10,135,45,45
11,127,42,42
12,87,29,29
14,66,22,22
1,65,22,22
0,56,19,19
5,23,8,8
13,20,6,7


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name           | Type               | Params | Mode 
--------------------------------------------------------------
0 | net            | Sequential         | 266 K  | train
1 | train_accuracy | MulticlassAccuracy | 0      | train
2 | val_accuracy   | MulticlassAccuracy | 0      | train
3 | test_accuracy  | MulticlassAccuracy | 0      | train
4 | val_f1         | MulticlassF1Score  | 0      | train
5 | test_f1        | MulticlassF1Score  | 0      | train
6 | criterion      | CrossEntropyLoss   | 0      | train
--------------------------------------------------------------
266 K     Trainable params
0         Non-trainable params
266 K     Total params
1.066     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode


=== Dataset metrics (all splits combined) ===
Total vocalizations: 1785
Maximum vocalizations per individual: 388
Minimum vocalizations per individual: 8
Vocalizations per individual (mean ± SD): 111.6 ± 116.2
Vocalizations per individual (mean ± SD, rounded): 112 ± 116


...

,individual,precision,recall,f1-score,support
0,32PC1_2021-22,0.947368,0.947368,0.947368,19.0
1,32PC1_2022-23,0.956522,1.000000,0.977778,22.0
2,AL_2019-20,1.000000,1.000000,1.000000,2.0
3,AL_2021-22,0.833333,1.000000,0.909091,5.0
4,BUG_2021-22,1.000000,0.833333,0.909091,6.0


Validation — Accuracy:          0.9748 (97.5%)
Validation — Balanced accuracy: 0.9627 (96.3%)
Validation — Recall (macro):    0.9627
Validation — Recall (weighted): 0.9748
Validation — F1 (macro):        0.9724 (97.2%)
Validation — F1 (weighted):     0.9746
Validation — ROC AUC (macro):   0.9996

Test — Accuracy:          0.9636 (96.4%)
Test — Balanced accuracy: 0.9477 (94.8%)
Test — Recall (macro):    0.9477
Test — Recall (weighted): 0.9636
Test — F1 (macro):        0.9508 (95.1%)
Test — F1 (weighted):     0.9634
Test — ROC AUC (macro):   0.9992

Metrics file appended: /teamspace/studios/this_studio/Results/rtbc/acrossyear/Metrics/rtbc_acrossyear_perch_gap_final_metrics.csv


,run_name,seed,dataset_name,split_type,model,aggegration,embedding_dim,hidden_dim,learning_rate,val_accuracy,...,val_f1_macro,val_f1_weighted,val_roc_auc_macro,test_accuracy,test_balanced_accuracy,test_recall_macro,test_recall_weighted,test_f1_macro,test_f1_weighted,test_roc_auc_macro
0,rtbc_acrossyear_perch_gap,13,rtbc,acrossyear,perch,gap,1280,256,0.001,0.977591,...,0.961901,0.977025,0.999766,0.955182,0.913476,0.913476,0.955182,0.931622,0.954603,0.998718
1,rtbc_acrossyear_perch_gap,18,rtbc,acrossyear,perch,gap,1280,256,0.001,0.974790,...,0.959225,0.974881,0.999375,0.966387,0.903573,0.903573,0.966387,0.928057,0.965571,0.999673
2,rtbc_acrossyear_perch_gap,46,rtbc,acrossyear,perch,gap,1280,256,0.001,0.969188,...,0.954585,0.968294,0.999501,0.957983,0.947155,0.947155,0.957983,0.958428,0.957676,0.998894
3,rtbc_acrossyear_perch_gap,123,rtbc,acrossyear,perch,gap,1280,256,0.001,0.974790,...,0.970662,0.974354,0.999437,0.969188,0.980130,0.980130,0.969188,0.983086,0.969148,0.999864
4,rtbc_acrossyear_perch_gap,321,rtbc,acrossyear,perch,gap,1280,256,0.001,0.974790,...,0.972366,0.974579,0.999588,0.963585,0.947727,0.947727,0.963585,0.950758,0.963440,0.999230



Aggregate summary across seeds:


,dataset_name,split_type,model,aggegration_mode,embedding_dim,n_seeds,test_accuracy,test_f1_macro,test_roc_auc_macro
0,rtbc,acrossyear,perch,gap,1280,5,0.962 ± 0.006,0.950 ± 0.022,0.9993 ± 0.0005
